In [2]:
import numpy as np
import pymc as pm
import arviz as az
import os
import re
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
from bs4 import BeautifulSoup
import pickle
from datetime import time, datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import numpy as np
from scipy.optimize import minimize

import numpy as np
from scipy.stats import norm
import random

random.seed(42)
import warnings
warnings.filterwarnings("ignore")

In [3]:
BAD_PIDS=['robas+120@developers.pg.com',
 'robas+171@developers.pg.com',
 'robas+146@developers.pg.com',
 'robas+157@developers.pg.com',
 'robas+170@developers.pg.com',
 'robas+161@developers.pg.com',
 'robas+180@developers.pg.com',
 'robas+173@developers.pg.com','robas+152@developers.pg.com']

In [9]:
script_dir = os.getcwd()
ROOT_PATH = os.path.join(script_dir, '../../OralyticsMRT/')

ORALYTICS_MRT_FILE = os.path.join(ROOT_PATH, "oralytics_mrt_data.csv")
MRT_PARTICIPANTS_IDS_FILE = os.path.join(ROOT_PATH, "mrt_participant_ids.csv")
RL_PATH_PREFIX = os.path.join(ROOT_PATH, "rl data tables/")
ACTION_SELECTION_TABLE = os.path.join(RL_PATH_PREFIX, "action_selection_data_table.csv")
USER_INFO_FILE = os.path.join(RL_PATH_PREFIX, "user_info_table.csv")
RAW_BRUSHING_INFO_FILE = os.path.join(ROOT_PATH, 'raw_brushing_info.html')
EXTRACTED_SEGMENTS_FILE = os.path.join(os.getcwd(), 'data_tables', 'extracted_device_segments.txt')

# Load datasets
anna_df = pd.read_csv(ORALYTICS_MRT_FILE)
anna_df = anna_df.sort_values(by=['user_id', 'decision_time'])
anna_df = anna_df[~anna_df['user_id'].isin(BAD_PIDS)]

mrt_df = pd.read_csv(MRT_PARTICIPANTS_IDS_FILE)
mrt_df = mrt_df[~mrt_df['LY Email'].isin(BAD_PIDS)]
mrt_id_mapping = dict(zip(mrt_df['LY ID'], mrt_df['LY Email']))
reverse_mrt_id_mapping = dict(zip(mrt_df['LY Email'], mrt_df['LY ID']))
MRT_LY_ID = list(reverse_mrt_id_mapping.keys())
# Load action selection table
ACTION_SELECTION = pd.read_csv(ACTION_SELECTION_TABLE)

# Load user information
USER_INFO_DF = pd.read_csv(USER_INFO_FILE)

# T0_PATH_PREFIX = os.path.join(ROOT_PATH, "Oralytics_simulation/data_tables/")
# T0_TABLE = os.path.join(T0_PATH_PREFIX, "user_t0_ranges.csv")
# T0_RANGE_DF = pd.read_csv(T0_TABLE)

In [4]:


USER_BRUSHING_PATH = os.path.join(T0_PATH_PREFIX, "user_brushing_info.pkl")
with open(USER_BRUSHING_PATH, 'rb') as file:
    all_user_brushing = pickle.load(file)


all_user_app_opening={}
for user_id in all_user_brushing.keys():
    USER_APP_OPENING_PATH = os.path.join(T0_PATH_PREFIX, "app_openingtime/")
    all_user_app_opening[user_id] = np.load(USER_APP_OPENING_PATH+user_id+'.npy',allow_pickle=True)

In [10]:
import os
import json
from datetime import datetime, date
from bisect import bisect_right
from datetime import timedelta
# Date range
start_date = date(2023, 12, 1)
end_date = date(2024, 2, 9)

def get_user_info(col_name, user_id):
    return USER_INFO_DF[USER_INFO_DF['user_id'] == user_id][col_name].values[0]

for user_id in all_user_brushing.keys():
    start_user_date = datetime.strptime(get_user_info("user_start_day", user_id), '%Y-%m-%d').date()
    end_user_date = datetime.strptime(get_user_info("user_end_day", user_id), '%Y-%m-%d').date()

    morning_decision_time=USER_INFO_DF[USER_INFO_DF['user_id'] == user_id]['morning_time_weekday'].values[0]
    morning_decision_time_weekday = datetime.strptime(morning_decision_time, '%H:%M:%S')
    morning_decision_time_weekday=morning_decision_time_weekday.time()

    morning_decision_time=USER_INFO_DF[USER_INFO_DF['user_id'] == user_id]['morning_time_weekend'].values[0]
    morning_decision_time_weekend = datetime.strptime(morning_decision_time, '%H:%M:%S')
    morning_decision_time_weekend=morning_decision_time_weekend.time()

    
    evening_decision_time=USER_INFO_DF[USER_INFO_DF['user_id'] == user_id]['evening_time_weekday'].values[0]
    evening_decision_time_weekday = datetime.strptime(evening_decision_time, '%H:%M:%S')
    evening_decision_time_weekday=evening_decision_time_weekday.time()

    evening_decision_time=USER_INFO_DF[USER_INFO_DF['user_id'] == user_id]['evening_time_weekend'].values[0]
    evening_decision_time_weekend = datetime.strptime(evening_decision_time, '%H:%M:%S')
    evening_decision_time_weekend=evening_decision_time_weekend.time()

    APP_DATA_PATH = os.path.join(ROOT_PATH, "main controller data", "app analytics data", f"app_data_{user_id}.json")
    try:
        with open(APP_DATA_PATH, 'r') as file:
            app_data = json.load(file)
    except FileNotFoundError:
        print(f"No app data file found for user {user_id}")
        continue



    records = []

    current_date = start_user_date
    while current_date <= end_user_date:
        is_weekend = current_date.weekday() >= 5  # 5 = Saturday, 6 = Sunday

        # Morning
        morning_time = morning_decision_time_weekend if is_weekend else morning_decision_time_weekday
        morning_dt = datetime.combine(current_date, morning_time)
        records.append({
            "date": current_date,
            "period": "morning",
            "decision_time": morning_dt,
            "action": 0,
            "message_type": None
        })

        # Evening
        evening_time = evening_decision_time_weekend if is_weekend else evening_decision_time_weekday
        evening_dt = datetime.combine(current_date, evening_time)
        records.append({
            "date": current_date,
            "period": "evening",
            "decision_time": evening_dt,
            "action": 0,
            "message_type": None
        })

        current_date += timedelta(days=1)
    df = pd.DataFrame(records)
    

    all_filtered = []
    app_scheduling_times=[]


    message_schedule_dict={}


    for record in app_data:
        if 'analytics_data' in record:
            try:
                analytics = json.loads(record['analytics_data'])
                success= False
                for entry in analytics:
                    event = entry.get("event", "")
                    if( 'GetScheduledMessagesSuccess' in event):
                        success= True
                        break
                if(not success):
                    continue
                for entry in analytics:
                    event = entry.get("event", "")
                    if event.startswith("ScheduledMessage:"):
                        try:
                            event_time=datetime.strptime(event.split(",")[1], "%Y-%m-%d %H:%M:%S")
                            event_time = event_time.replace(second=0, microsecond=0)
                            if start_user_date <= event_time.date() <= end_user_date:
                                app_time=entry.get("app_created_at","")
                                app_time=datetime.strptime(app_time, "%Y-%m-%d %H:%M:%S")
                                app_time = app_time.replace(second=0, microsecond=0)
                                if(app_time not in message_schedule_dict):
                                    message_schedule_dict[app_time]=[]
                                if(app_time <= event_time):
                                    message_schedule_dict[app_time].append([event_time,event.split(",")[0].split(":")[1]])
                        except (IndexError, ValueError):
                            continue
            except json.JSONDecodeError:
                print(f"Failed to parse analytics_data for email {record.get('email')}")

    sorted_message_times = sorted(message_schedule_dict.keys())

    
    updated_records = []
    #print("\n")
    for idx, row in df.iterrows():
        decision_time = row['decision_time']


        matched_time = None
        min_diff = timedelta.max  # Start with maximum possible difference

        for t in sorted_message_times:
            if t <= decision_time:
                diff = decision_time - t
                if diff < min_diff:
                    min_diff = diff
                    matched_time = t
            else:
                break  # Since the list is sorted, stop early


        message_type = None
        action = 0
        if matched_time:
            for msg_time, msg_type in message_schedule_dict[matched_time]:
                if (msg_time==row['decision_time']):
                    message_type=msg_type
                    action=1
                    break
        updated_records.append({
            **row,
            "matched_app_time": matched_time,
            "action": action,
            "message_type": message_type
        })

    df_updated = pd.DataFrame(updated_records)

    print(len(df_updated[df_updated['action']==1]),user_id)

    user_actions_anna = (
    anna_df[anna_df['user_id'] == user_id]
    .dropna(subset=['user_start_day'])
    .reset_index()
    )
    user_actions_anna['decision_time'] = pd.to_datetime(user_actions_anna['decision_time'], format='%Y-%m-%d %H:%M:%S')
    user_actions_anna = user_actions_anna.sort_values(by='decision_time')
    print(len(df_updated[df_updated['action']==1]),user_id)
    

    

NameError: name 'all_user_brushing' is not defined